In [1]:
%pip install -q pixeltable-pgserver "psycopg[binary]" sqlalchemy pandas kagglehub

import pandas as pd
import pixeltable_pgserver as pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

/usr/local/lib/python3.13/dist-packages/platformdirs/api.py:467: RuntimeDirWarning: XDG_RUNTIME_DIR is not set, falling back to /tmp/runtime-0
  return Path(self.user_runtime_dir)


PostgreSQL 16.15 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-11), 64-bit


In [2]:
import kagglehub

csv_files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}
dataset_dir = None
try:
    candidate = Path(kagglehub.dataset_download('olistbr/brazilian-ecommerce'))
    paths = list(candidate.rglob('olist_customers_dataset.csv'))
    if not paths:
        raise FileNotFoundError('CSV Olist не знайдено у завантаженому наборі')
    candidate = paths[0].parent
    for filename in csv_files.values():
        if not (candidate / filename).exists():
            raise FileNotFoundError(filename)
    dataset_dir = candidate
    print('Використовується реальний Olist:', dataset_dir)
except Exception as exc:
    print('Kaggle недоступний, використовуємо stub fallback:', str(exc)[:300])


Kaggle недоступний, використовуємо stub fallback: name 'Path' is not defined


In [3]:
if dataset_dir is not None:
    for table_name, file_name in csv_files.items():
        df = pd.read_csv(dataset_dir / file_name)
        df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
        print(f'{table_name}: {len(df):,} рядків')
else:
    print('Raw-шар пропущено: використовується stub-schema fallback')


Raw-шар пропущено: використовується stub-schema fallback


In [4]:
# Видаляємо похідні об'єкти до перебудови базових таблиць.
cleanup = [
    "DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE",
    "DROP TABLE IF EXISTS hw5_referrals CASCADE",
]
with engine.begin() as conn:
    for stmt in cleanup:
        conn.execute(text(stmt))
print("Похідні об'єкти очищено")


Похідні об'єкти очищено


In [5]:
statements = [
    """
DROP TABLE IF EXISTS olist_order_items CASCADE;
""",
    """
DROP TABLE IF EXISTS olist_orders      CASCADE;
""",
    """
DROP TABLE IF EXISTS olist_customers   CASCADE;
""",
    """
DROP TABLE IF EXISTS olist_products    CASCADE;
""",
    """
DROP TABLE IF EXISTS olist_sellers     CASCADE;
""",
    """
CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);
""",
    """
CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);
""",
    """
CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);
""",
    """
CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);
""",
    """
CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
"""
]
with engine.begin() as conn:
    for statement in statements:
        conn.execute(text(statement))
print(f'Виконано SQL-команд: {len(statements)}')


Виконано SQL-команд: 10


In [6]:
if dataset_dir is not None:
    print('Режим: Olist CSV')
else:
    print('Режим: stub-schema fallback')


Режим: stub-schema fallback


In [7]:
real_statements = [
    """
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;
""",
    """
INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;
""",
    """
INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;
""",
    """
INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;
""",
    """
INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;
"""
]
stub_statements = [
    """
INSERT INTO olist_sellers (seller_id, seller_zip_code_prefix, seller_city, seller_state)
VALUES
    ('s_01', 10001, 'Sao Paulo', 'SP'),
    ('s_02', 20001, 'Rio de Janeiro', 'RJ'),
    ('s_03', 30001, 'Belo Horizonte', 'MG');
""",
    """
INSERT INTO olist_products (
    product_id, product_category_name,
    product_name_length, product_description_length,
    product_photos_qty, product_weight_g,
    product_length_cm, product_height_cm, product_width_cm
)
VALUES
    ('p_01', 'electronics', 10, 100, 2,  800, 20, 10, 15),
    ('p_02', 'electronics', 12, 120, 3, 1200, 25, 12, 18),
    ('p_03', 'home',        8,  90, 1, 1500, 30, 20, 25),
    ('p_04', 'home',        9,  80, 2,  700, 18, 10, 14),
    ('p_05', 'fashion',     7,  60, 1,  300, 10,  5,  8),
    ('p_06', 'beauty',      6,  50, 1,  200,  8,  4,  6),
    ('p_07', 'sport',       9,  70, 2,  900, 20, 15, 16);
""",
    """
INSERT INTO olist_customers (
    customer_id, customer_unique_id, customer_zip_code_prefix,
    customer_state, customer_city
)
SELECT
    'c_' || LPAD(n::TEXT, 3, '0'),
    'u_' || LPAD(((n - 1) % 25 + 1)::TEXT, 3, '0'),
    10000 + n,
    (ARRAY['SP', 'RJ', 'MG', 'BA', 'PR'])[(n % 5) + 1],
    'City' || (n % 10)
FROM generate_series(1, 30) AS n;
""",
    """
INSERT INTO olist_orders (
    order_id, customer_id, order_status,
    order_purchase_timestamp, order_approved_at,
    order_delivered_carrier_date, order_delivered_customer_date,
    order_estimated_delivery_date
)
SELECT
    'o_' || LPAD(n::TEXT, 3, '0'),
    'c_' || LPAD(((n % 30) + 1)::TEXT, 3, '0'),
    (ARRAY['delivered', 'delivered', 'delivered', 'canceled', 'shipped'])[(n % 5) + 1],
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days'),
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days') + INTERVAL '1 day',
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days') + INTERVAL '3 days',
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days') + INTERVAL '8 days',
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days') + INTERVAL '10 days'
FROM generate_series(1, 50) AS n;
""",
    """
INSERT INTO olist_order_items (
    order_id, order_item_id, product_id, seller_id,
    shipping_limit_date, price, freight_value
)
SELECT
    'o_' || LPAD(((n - 1) % 50 + 1)::TEXT, 3, '0'),
    ((n - 1) / 50) + 1,
    'p_0' || ((n % 7) + 1),
    's_0' || ((n % 3) + 1),
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days') + INTERVAL '5 days',
    (n % 200 + 10)::NUMERIC(12, 2),
    (n % 30 + 5)::NUMERIC(12, 2)
FROM generate_series(1, 100) AS n;
"""
]
statements = real_statements if dataset_dir is not None else stub_statements
with engine.begin() as conn:
    for statement in statements:
        conn.execute(text(statement))
print(f'Виконано INSERT-команд: {len(statements)}')


Виконано INSERT-команд: 5


In [8]:
query = """
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(15))


,table_name,n_rows
0,customers,30
1,order_items,100
2,orders,50
3,products,7
4,sellers,3


In [9]:
query = """
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 20;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(20))


,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,c_021,RJ,1,2,360.0,2024-05-20,2024-04-01,210.0,0.0,electronics,180.0
1,c_020,SP,8,2,352.0,2024-05-13,2024-04-01,206.0,0.0,electronics,206.0
2,c_019,PR,15,2,344.0,2024-05-06,2024-04-01,202.0,1.0,electronics,121.0
3,c_018,BA,22,2,336.0,2024-04-29,2024-04-01,198.0,0.0,sport,119.0
4,c_010,SP,78,2,332.0,2024-03-04,2024-01-01,196.0,0.0,home,136.0
5,c_017,MG,29,2,328.0,2024-04-22,2024-04-01,194.0,0.0,home,134.0
6,c_009,PR,85,2,324.0,2024-02-26,2024-01-01,192.0,1.0,home,162.0
7,c_016,RJ,36,2,320.0,2024-04-15,2024-04-01,190.0,0.0,home,160.0
8,c_008,BA,92,2,316.0,2024-02-19,2024-01-01,188.0,0.0,home,188.0
9,c_015,SP,43,2,312.0,2024-04-08,2024-04-01,186.0,0.0,home,186.0


In [10]:
query = """
SELECT COUNT(*) AS customers, COUNT(DISTINCT customer_id) AS unique_customers FROM olist_customers;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(15))


,customers,unique_customers
0,30,30


In [11]:
statements = [
    """
DROP TABLE IF EXISTS hw5_referrals CASCADE;
""",
    """
CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);
""",
    """
WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    prev.customer_id AS referrer_customer_id,
    nxt.customer_id AS referred_customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt
    ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;
"""
]
with engine.begin() as conn:
    for statement in statements:
        conn.execute(text(statement))
print(f'Виконано SQL-команд: {len(statements)}')


Виконано SQL-команд: 3


In [12]:
query = """
SELECT * FROM hw5_referrals ORDER BY referrer_customer_id, referred_customer_id;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(15))


,referrer_customer_id,referred_customer_id,referred_at
0,c_001,c_002,2026-11-08 12:43:03.719880+00:00
1,c_002,c_003,2026-11-08 12:43:03.719880+00:00
2,c_003,c_004,2026-11-08 12:43:03.719880+00:00
3,c_004,c_005,2026-11-08 12:43:03.719880+00:00
4,c_005,c_006,2026-11-08 12:43:03.719880+00:00
5,c_006,c_007,2026-11-08 12:43:03.719880+00:00
6,c_007,c_008,2026-11-08 12:43:03.719880+00:00
7,c_008,c_009,2026-11-08 12:43:03.719880+00:00
8,c_009,c_010,2026-11-08 12:43:03.719880+00:00


In [13]:
query = """
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: root customers, які не були referred іншими customers
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: перейти від referrer до його referred customers
    SELECT
        r.referred_customer_id AS customer_id,
        rd.depth + 1 AS depth,
        rd.path_ids || r.referred_customer_id AS path_ids,
        rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)
SELECT
    depth,
    COUNT(*) AS n_customers,
    MAX(path_text) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(50))


,depth,n_customers,sample_path
0,1,21,c_030
1,2,1,c_001 -> c_002
2,3,1,c_001 -> c_002 -> c_003
3,4,1,c_001 -> c_002 -> c_003 -> c_004
4,5,1,c_001 -> c_002 -> c_003 -> c_004 -> c_005
5,6,1,c_001 -> c_002 -> c_003 -> c_004 -> c_005 -> c...
6,7,1,c_001 -> c_002 -> c_003 -> c_004 -> c_005 -> c...
7,8,1,c_001 -> c_002 -> c_003 -> c_004 -> c_005 -> c...
8,9,1,c_001 -> c_002 -> c_003 -> c_004 -> c_005 -> c...
9,10,1,c_001 -> c_002 -> c_003 -> c_004 -> c_005 -> c...


In [14]:
statements = [
    """
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;
"""
]
with engine.begin() as conn:
    for statement in statements:
        conn.execute(text(statement))
print(f'Виконано SQL-команд: {len(statements)}')


Виконано SQL-команд: 1


In [15]:
query = """
CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id;
"""
with engine.begin() as conn:
    conn.execute(text(query))
print('SQL виконано')


SQL виконано


In [16]:
query = """
CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id ON mv_olist_customer_features(customer_id);
"""
with engine.begin() as conn:
    conn.execute(text(query))
print('SQL виконано')


SQL виконано


In [17]:
query = """
SELECT COUNT(*) AS n_rows, COUNT(DISTINCT customer_id) AS n_unique FROM mv_olist_customer_features;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(15))


,n_rows,n_unique
0,30,30


In [18]:
query = """
SELECT * FROM mv_olist_customer_features ORDER BY monetary DESC NULLS LAST, customer_id LIMIT 5;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(5))


,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,c_021,RJ,1,2,360.0,2024-05-20,2024-04-01,210.0,0.0,electronics,180.0
1,c_020,SP,8,2,352.0,2024-05-13,2024-04-01,206.0,0.0,electronics,206.0
2,c_019,PR,15,2,344.0,2024-05-06,2024-04-01,202.0,1.0,electronics,121.0
3,c_018,BA,22,2,336.0,2024-04-29,2024-04-01,198.0,0.0,sport,119.0
4,c_010,SP,78,2,332.0,2024-03-04,2024-01-01,196.0,0.0,home,136.0


In [19]:
query = """
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;
"""
with engine.begin() as conn:
    conn.execute(text(query))
print('SQL виконано')


SQL виконано


In [20]:
query = """
SELECT
    (SELECT COUNT(*) FROM olist_customers) AS customers,
    (SELECT COUNT(*) FROM mv_olist_customer_features) AS mv_rows,
    (SELECT COUNT(DISTINCT customer_id) FROM mv_olist_customer_features) AS unique_mv_customers;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(15))


,customers,mv_rows,unique_mv_customers
0,30,30,30


In [21]:
query = """
EXPLAIN (ANALYZE, BUFFERS)
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id;
"""
with engine.connect() as conn:
    plan = conn.execute(text(query)).scalars().all()
print('\n'.join(plan))


Hash Left Join  (cost=317.39..339.49 rows=580 width=158) (actual time=1.553..1.697 rows=30 loops=1)
  Hash Cond: (c.customer_id = ranked.customer_id)
  Buffers: shared hit=7
  CTE stage1_orders_total
    ->  HashAggregate  (cost=40.77..49.35 rows=490 width=124) (actual time=0.218..0.282 rows=50 loops=1)
          Group Key: o_1.order_id
          Batches: 1  Memory Usage: 73kB
          Buffers: shared hit=2
          ->  Hash Right Join  (cost=21.02..37.10 rows=490 width=136) (actual time=0.059..0.129 rows=100 loops=1)
                Hash Cond: (oi_1.order_id = o_1.order_id)
                Buffers: shared hit=2
                ->  Seq Scan on olist_order_items oi_1  (cost=0.00..14.80 rows=480 width=64) (actual time=0.007..0.019 rows=100 loops=1)
                      Buffers: shared hit=1
                ->  Hash  (cost=14.90..14.90 rows=490 width=104) (actual time=0.036..0.036 rows=50 loops=1)
                      Buckets: 1024  Batches: 1  Memory Usage: 12kB
                     

In [22]:
query = """
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_id = mv.customer_id;
"""
with engine.connect() as conn:
    plan = conn.execute(text(query)).scalars().all()
print('\n'.join(plan))


Hash Join  (cost=0.26..1.68 rows=1 width=158) (actual time=0.063..0.072 rows=1 loops=1)
  Hash Cond: (mv.customer_id = olist_customers.customer_id)
  Buffers: shared hit=3
  ->  Seq Scan on mv_olist_customer_features mv  (cost=0.00..1.30 rows=30 width=158) (actual time=0.010..0.013 rows=30 loops=1)
        Buffers: shared hit=1
  ->  Hash  (cost=0.25..0.25 rows=1 width=32) (actual time=0.040..0.041 rows=1 loops=1)
        Buckets: 1024  Batches: 1  Memory Usage: 9kB
        Buffers: shared hit=2
        ->  Limit  (cost=0.15..0.25 rows=1 width=32) (actual time=0.036..0.036 rows=1 loops=1)
              Buffers: shared hit=2
              ->  Index Only Scan using olist_customers_pkey on olist_customers  (cost=0.15..56.85 rows=580 width=32) (actual time=0.034..0.035 rows=1 loops=1)
                    Heap Fetches: 1
                    Buffers: shared hit=2
Planning Time: 0.824 ms
Execution Time: 0.118 ms


In [23]:
query = """
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary ON mv_olist_customer_features (monetary DESC);
"""
with engine.begin() as conn:
    conn.execute(text(query))
print('SQL виконано')


SQL виконано


In [24]:
query = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10;
"""
with engine.connect() as conn:
    plan = conn.execute(text(query)).scalars().all()
print('\n'.join(plan))


Limit  (cost=1.95..1.97 rows=10 width=158) (actual time=0.038..0.041 rows=10 loops=1)
  Buffers: shared hit=1
  ->  Sort  (cost=1.95..2.02 rows=30 width=158) (actual time=0.037..0.038 rows=10 loops=1)
        Sort Key: monetary DESC NULLS LAST
        Sort Method: top-N heapsort  Memory: 27kB
        Buffers: shared hit=1
        ->  Seq Scan on mv_olist_customer_features  (cost=0.00..1.30 rows=30 width=158) (actual time=0.009..0.012 rows=30 loops=1)
              Buffers: shared hit=1
Planning:
  Buffers: shared hit=15 read=1
Planning Time: 0.333 ms
Execution Time: 0.063 ms
